# 01 — Baseline VLM
Notebook de départ : charger une image, appliquer un prompt et sauvegarder une sortie JSON.

In [ ]:
# autoreload : recharge automatiquement src/*.py après une édition, sans redémarrer le kernel.
# (À placer avant tout import de `src` pour que les modules rechargés soient pris en compte.)
%load_ext autoreload
%autoreload 2

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE" # Cette commande permet d'éviter l'erreur "OMP: Error #15: Initializing libiomp5md.dll, but found libiomp5md.dll already initialized." lors de l'utilisation de certaines bibliothèques comme PyTorch ou TensorFlow sur Windows.

import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))

from src.inference_medgemma import medgemma_predict
from src.guardrails import apply_safety_guardrails

sample = Path('../data/sample_images/CXR_SYN_002_suspected_opacity.png')
result = apply_safety_guardrails(medgemma_predict(sample, mode='baseline'))

In [2]:
result

{'image_quality': 'mauvaise',
 'predicted_class': 'uncertain',
 'confidence': 0.0,
 'visual_evidence': '',
 'justification': 'Sortie non parsable.',
 'limitations': 'JSON invalide renvoyé par le modèle.',
 'warning': 'Prototype pédagogique. Non destiné au diagnostic. Validation par un professionnel qualifié requise.',
 '_raw': '```json\n{"image_quality":"bonne","predicted_class":"normal","confidence":0.9,"visual_evidence":"L\'image est nette et bien définie. Les structures pulmonaires sont visibles. Il n\'y a pas de signes évidents d\'opacité ou de pneumothorax.", "justification":"L\'image montre une radiographie thoracique frontale avec une bonne qualité d\'image. Les structures pulmonaires sont bien visibles et il n\'y a pas de signes d\'opacité ou de pneumothorax.", "limitations":"L\'interprétation est limitée par la qualité de l\'image et',
 'model_name': 'google/medgemma-4b-it (cuda/bf16)',
 'prompt_version': 'baseline_v1',
 'latency_ms': 15605,
 'guardrail_errors': []}

### Testons maintenant avec une vraie image issue du dataset RSNA

In [3]:
sample2 = Path('../data/RSNA_Pneumonia/Images/00a85be6-6eb0-421d-8acf-ff2dc0007e8a.png')
result2 = apply_safety_guardrails(medgemma_predict(sample2, mode='baseline'))

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [4]:
result2

{'image_quality': 'mauvaise',
 'predicted_class': 'uncertain',
 'confidence': 0.0,
 'visual_evidence': '',
 'justification': 'Sortie non parsable.',
 'limitations': 'JSON invalide renvoyé par le modèle.',
 'warning': 'Prototype pédagogique. Non destiné au diagnostic. Validation par un professionnel qualifié requise.',
 '_raw': '```json\n{"image_quality":"bonne","predicted_class":"normal","confidence":0.9,"visual_evidence":"Les poumons apparaissent clairs, sans opacités évidentes. Le cœur et les grandes vaisseaux sont bien visibles. La colonne vertébrale est intacte.","justification":"L\'image est bien exposée et les structures anatomiques sont clairement visibles. Il n\'y a pas de signes d\'opacité significative.","limitations":"L\'analyse est basée uniquement sur l\'image et ne peut pas exclure des pathologies subtiles.","warning":"Une évaluation clinique est nécessaire pour',
 'model_name': 'google/medgemma-4b-it (cuda/bf16)',
 'prompt_version': 'baseline_v1',
 'latency_ms': 14432,
 

## Bon usage de la pipeline : `medgemma_predict` → guardrails → `summarize_metrics`

On évalue le baseline sur **20 images du dataset RSNA Pneumonia**.

> RSNA_Pneumonia n'a pas de CSV de labels, mais le dossier `Masks/` marque les zones d'opacité.
> On dérive donc la vérité terrain depuis les masques : **masque non vide → `suspected_opacity`**,
> **masque vide → `normal`**. On prend un échantillon équilibré (10 + 10).

In [ ]:
import numpy as np
from PIL import Image
from src.guardrails import apply_safety_guardrails
from src.metrics import summarize_metrics, confusion_counts

IMG_DIR = Path('../data/RSNA_Pneumonia/Images')
MASK_DIR = Path('../data/RSNA_Pneumonia/Masks')

def label_from_mask(name: str) -> str:
    """Vérité terrain dérivée du masque : opacité présente -> suspected_opacity, sinon normal."""
    arr = np.array(Image.open(MASK_DIR / name).convert('L'))
    return 'suspected_opacity' if arr.max() > 0 else 'normal'

# Échantillon équilibré : 10 images avec opacité + 10 normales
all_names = sorted(p.name for p in IMG_DIR.glob('*.png'))
opacity = [n for n in all_names if label_from_mask(n) == 'suspected_opacity'][:10]
normal  = [n for n in all_names if label_from_mask(n) == 'normal'][:10]
sample_names = opacity + normal
print(f'{len(sample_names)} images sélectionnées : {len(opacity)} opacity + {len(normal)} normal')

In [ ]:
# Boucle d'inférence : medgemma_predict -> apply_safety_guardrails -> ligne d'évaluation
# (même pattern que eval/run_evaluation.py). ~15 s/image sur GPU -> ~5 min pour 20 images.
rows = []
for i, name in enumerate(sample_names, 1):
    pred = apply_safety_guardrails(medgemma_predict(IMG_DIR / name, mode='baseline'))
    row = {
        'case_id': name,
        'label': label_from_mask(name),      # vérité terrain (masque)
        'predicted_class': pred['predicted_class'],
        'confidence': pred['confidence'],
        'json_valid': pred['json_valid'],    # posé par medgemma_predict ; KeyError = kernel périmé
        'warning': pred.get('warning', ''),
        'latency_ms': pred.get('latency_ms', 0),
    }
    rows.append(row)
    print(f"[{i:2d}/20] {name[:8]}  vrai={row['label']:17s} pred={row['predicted_class']:17s} "
          f"json_valid={row['json_valid']}  {row['latency_ms']} ms")

In [ ]:
# Agrégation : les deux fonctions de metrics.py
metrics = summarize_metrics(rows)
print('json_valid_rate :', metrics['json_valid_rate'], '  (reflète désormais les vrais échecs de parsing)')
print('accuracy        :', metrics['accuracy'])
print('macro_f1        :', metrics['macro_f1'])
print('avg_latency_ms  :', metrics['avg_latency_ms'])
print('uncertain_rate  :', metrics['uncertain_rate'])
print('all_targets_met :', metrics['all_targets_met'])
print('\nMatrice de confusion (vrai__prédit) :')
confusion_counts([r['label'] for r in rows], [r['predicted_class'] for r in rows])